# EDA on Retail Sales Data
**OASIS INFOBYTE — Data Analytics, Level 1, Task 1**

This notebook inspects retail transactions, cleans missing values, summarizes numerical data, and creates visualizations for sales trends, demographics, products, categories, correlations, and weekday patterns.

> **Dataset notice:** The included CSV is synthetic demonstration data created for this project. Replace it with an approved real retail dataset before final submission. Do not present these sample-data findings as real business results.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "retail_sales_sample.csv"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
print("Project folder:", ROOT)
print("Dataset:", DATA_PATH)

## 1. Load and inspect the dataset
We first check the number of rows and columns, column types, missing values, and duplicate rows. This helps identify data-quality problems before analysis.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn data types:")
display(df.dtypes.to_frame("Data type"))
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing count"))
print("\nDuplicate rows:", df.duplicated().sum())

**Observation:** Review the printed shape and missing-value counts. The provided sample intentionally contains a few missing values so the cleaning step can be demonstrated.

## 2. Clean data and create analysis features
Remove duplicate records, fill missing numerical values with medians, label missing categories, and create month, quarter, and age-group fields.

In [ ]:
df = df.drop_duplicates().copy()
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce")
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Gender"] = df["Gender"].fillna("Not specified")
df["Product"] = df["Product"].fillna("Unknown product")
df["Unit_Price"] = df["Unit_Price"].fillna(df["Unit_Price"].median())
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce").fillna(1)
df["Total_Amount"] = df["Quantity"] * df["Unit_Price"]
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)
df["Age_Group"] = pd.cut(df["Age"], bins=[17,24,34,44,54,64,100],
                         labels=["18–24","25–34","35–44","45–54","55–64","65+"])
print("Missing values after cleaning:")
display(df.isna().sum().to_frame("Missing count"))

## 3. Descriptive statistics
Mean, median, mode, and standard deviation describe typical values and spread for age, quantity, unit price, and transaction amount.

In [ ]:
numeric_cols = ["Age", "Quantity", "Unit_Price", "Total_Amount"]
display(df[numeric_cols].describe().T)
stats = pd.DataFrame({
    "Mean": df[numeric_cols].mean(),
    "Median": df[numeric_cols].median(),
    "Mode": df[numeric_cols].mode().iloc[0],
    "Standard deviation": df[numeric_cols].std()
})
display(stats.round(2))

**Observation:** Use the tables to identify typical transaction size and variation. For final submission, describe the values from your chosen real dataset.

## 4. Monthly and quarterly sales trends
Line charts help reveal seasonal changes, growth, or unusually high/low sales periods.

In [ ]:
monthly = df.groupby("Month")["Total_Amount"].sum()
ax = monthly.plot(figsize=(12,5), marker="o", title="Monthly Retail Sales")
ax.set_xlabel("Month"); ax.set_ylabel("Revenue"); plt.xticks(rotation=60); plt.tight_layout()
plt.savefig(OUT / "monthly_sales.png", dpi=160); plt.show()

quarterly = df.groupby("Quarter")["Total_Amount"].sum()
ax = quarterly.plot(figsize=(9,5), marker="o", title="Quarterly Retail Sales")
ax.set_xlabel("Quarter"); ax.set_ylabel("Revenue"); plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(OUT / "quarterly_sales.png", dpi=160); plt.show()

**Observation:** Note the strongest and weakest months/quarters. Do not claim seasonality unless the trend is consistent and supported by the data.

## 5. Customer demographics
Compare transaction counts across age groups and gender. These are transaction-level counts, not necessarily counts of unique customers.

In [ ]:
age_order = ["18–24","25–34","35–44","45–54","55–64","65+"]
plt.figure(figsize=(8,5))
df["Age_Group"].value_counts().reindex(age_order, fill_value=0).plot(kind="bar")
plt.title("Customer Age Group Distribution"); plt.xlabel("Age group"); plt.ylabel("Transactions")
plt.tight_layout(); plt.savefig(OUT / "age_group_distribution.png", dpi=160); plt.show()

plt.figure(figsize=(7,5))
df["Gender"].value_counts().plot(kind="bar")
plt.title("Transactions by Gender"); plt.xlabel("Gender"); plt.ylabel("Transactions")
plt.tight_layout(); plt.savefig(OUT / "gender_breakdown.png", dpi=160); plt.show()

**Observation:** Identify the most common age group and gender category. If customer-level behavior is needed, count unique `Customer_ID` values rather than transactions.

## 6. Product and category analysis
Rank products by units sold and compare total revenue across product categories.

In [ ]:
top_products = df.groupby("Product")["Quantity"].sum().sort_values(ascending=False).head(10)
ax = top_products.sort_values().plot(kind="barh", figsize=(10,6), title="Top 10 Best-Selling Products by Quantity")
ax.set_xlabel("Units sold"); ax.set_ylabel("Product"); plt.tight_layout()
plt.savefig(OUT / "top_10_products.png", dpi=160); plt.show()

category_revenue = df.groupby("Category")["Total_Amount"].sum().sort_values(ascending=False)
ax = category_revenue.plot(kind="bar", figsize=(9,5), title="Revenue by Product Category")
ax.set_xlabel("Category"); ax.set_ylabel("Revenue"); plt.xticks(rotation=25, ha="right"); plt.tight_layout()
plt.savefig(OUT / "revenue_by_category.png", dpi=160); plt.show()
display(category_revenue.to_frame("Revenue").round(2))

**Observation:** The best-selling product by quantity may not be the product with the highest revenue. Use both views when discussing inventory and promotions.

## 7. Correlation heatmap
Correlation ranges from -1 to +1. Correlation does not prove that one variable causes another.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[["Age","Quantity","Unit_Price","Total_Amount"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix"); plt.tight_layout()
plt.savefig(OUT / "correlation_heatmap.png", dpi=160); plt.show()

**Observation:** Check which numerical variables move together. Total amount is mathematically related to quantity and unit price, so correlations with total amount should be interpreted carefully.

## 8. Additional visualization: weekday average transaction value
This extra chart compares the average transaction amount by weekday to look for differences in shopping patterns.

In [ ]:
df["Weekday"] = df["Date"].dt.day_name()
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_avg = df.groupby("Weekday")["Total_Amount"].mean().reindex(weekday_order)
ax = weekday_avg.plot(kind="bar", figsize=(9,5), title="Average Transaction Value by Weekday")
ax.set_xlabel("Weekday"); ax.set_ylabel("Average transaction value")
plt.xticks(rotation=25, ha="right"); plt.tight_layout()
plt.savefig(OUT / "weekday_average_transaction.png", dpi=160); plt.show()
display(weekday_avg.round(2).to_frame("Average transaction value"))

**Observation:** Compare weekdays and weekends, but treat small differences cautiously. Validate the pattern with a real dataset before making staffing or promotion decisions.

## 9. Key findings and actionable recommendations
The cell below prints findings calculated from the loaded data. These are demo-only when using the bundled synthetic CSV.

In [ ]:
print(f"Total revenue: {df['Total_Amount'].sum():,.2f}")
print(f"Average transaction value: {df['Total_Amount'].mean():,.2f}")
print(f"Top revenue category: {category_revenue.index[0]} ({category_revenue.iloc[0]:,.2f})")
print(f"Best-selling product by units: {top_products.index[0]} ({int(top_products.iloc[0])} units)")
print(f"Largest age group by transaction: {df['Age_Group'].value_counts().idxmax()}")

print("\nRecommendations to validate with real data:")
print("1. Prioritize inventory and promotions for the category with the highest revenue.")
print("2. Feature best-selling products in bundles and product recommendations.")
print("3. Test age-group and weekday-specific promotions, then compare conversion and revenue.")

## 10. Export cleaned data and summary tables

In [ ]:
df.to_csv(OUT / "cleaned_retail_sales.csv", index=False)
category_revenue.rename("Revenue").to_csv(OUT / "category_revenue_summary.csv")
top_products.rename("Units_Sold").to_csv(OUT / "top_products_summary.csv")
print("Saved cleaned data and summary tables to:", OUT.resolve())